# Oxford Flowers 102 — Transfer Learning (v2, improved)
MobileNetV2 vs ResNet50V2 | Two-phase training | Contest submission

**Changes from v1 (see ANALYSIS_AND_RECOMMENDATIONS.md):**
1. BatchNorm layers kept **frozen** during fine-tuning (fixes the accuracy crash).
2. Unfreeze more backbone layers (40 MobileNet / 50 ResNet) with a higher fine-tune LR (1e-4).
3. Stronger augmentation + regularized head (BatchNorm, 2x Dropout, L2).
4. `ReduceLROnPlateau` + monitor `val_loss` (more stable than val_accuracy).

## 1. Install & Import

In [3]:
# !pip install -q tensorflow-datasets

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import os

os.makedirs('outputs', exist_ok=True)
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

ModuleNotFoundError: No module named 'tensorflow'


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\SDC13\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## 2. Load Dataset

In [ ]:
(ds_train, ds_val, ds_test), info = tfds.load(
    'oxford_flowers102',
    split=['train', 'validation', 'test'],
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = 102
IMG_SIZE    = 224
BATCH_SIZE  = 32

print('Train:', info.splits['train'].num_examples)
print('Val:  ', info.splits['validation'].num_examples)
print('Test: ', info.splits['test'].num_examples)

## 3. Data Pipeline
Stronger augmentation than v1 to fight overfitting (only 10 images/class).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

def make_pipeline(ds, preprocess_fn, training=False):
    def prep(img, label):
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32)
        img = preprocess_fn(img)
        return img, label

    ds = ds.map(prep, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1024)
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

## 4. Build Model (shared function)
Regularized head: BatchNorm + two Dropouts + L2 to reduce the train/val gap.

In [ ]:
def build_model(backbone):
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = backbone(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(256, activation='relu',
                              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)

## 5. Train & Evaluate (helper)
**Key fix:** during fine-tuning we unfreeze the top layers but keep every BatchNorm layer frozen, so the backbone's running statistics are not corrupted by the tiny batches.

In [ ]:
def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=6, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    ]


def train_and_evaluate(name, backbone, preprocess_fn, train_ds_raw, val_ds_raw,
                       fine_tune_layers=40):
    print(f'\n===============================')
    print(f'  {name}')
    print(f'===============================')

    train_ds = make_pipeline(train_ds_raw, preprocess_fn, training=True)
    val_ds   = make_pipeline(val_ds_raw,   preprocess_fn, training=False)

    model = build_model(backbone)

    # --- Phase 1: train head only (backbone frozen) ---
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    print('Phase 1: head training...')
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=20,
                   callbacks=make_callbacks())

    # --- Phase 2: fine-tune top layers, KEEP BatchNorm FROZEN ---
    backbone.trainable = True
    for layer in backbone.layers[:-fine_tune_layers]:
        layer.trainable = False
    for layer in backbone.layers:                       # <-- critical fix
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),       # higher than v1's 1e-5
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    print('Phase 2: fine-tuning...')
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=20,
                   callbacks=make_callbacks())

    # restore_best_weights already left `model` at its best val_loss point
    model.save(f'outputs/best_{name}.keras')
    best = model

    # --- Metrics on val set ---
    y_true, y_pred = [], []
    for imgs, labels in val_ds:
        preds = best.predict(imgs, verbose=0)
        y_pred.extend(np.argmax(preds, axis=1))
        y_true.extend(labels.numpy())
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    metrics = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1':        f1_score(y_true, y_pred, average='macro', zero_division=0),
    }
    print(f'\nVal Results for {name}:')
    for k, v in metrics.items():
        print(f'  {k:10s}: {v:.4f}')

    # --- Training curves ---
    acc  = h1.history['accuracy']     + h2.history['accuracy']
    vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    split = len(h1.history['accuracy'])

    plt.figure(figsize=(8, 4))
    plt.plot(acc,  label='Train acc')
    plt.plot(vacc, label='Val acc')
    plt.axvline(split, color='gray', linestyle='--', label='Fine-tune start')
    plt.title(f'{name} — Training Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'outputs/curves_{name}.png')
    plt.show()

    return best, metrics, y_true, y_pred

## 6. Train MobileNetV2

In [ ]:
mobilenet_backbone = tf.keras.applications.MobileNetV2(
    include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
mobilenet_preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

model_mobile, metrics_mobile, y_true_mobile, y_pred_mobile = train_and_evaluate(
    'MobileNetV2', mobilenet_backbone, mobilenet_preprocess, ds_train, ds_val,
    fine_tune_layers=40
)

## 7. Train ResNet50V2

In [ ]:
resnet_backbone = tf.keras.applications.ResNet50V2(
    include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
resnet_preprocess = tf.keras.applications.resnet_v2.preprocess_input

model_resnet, metrics_resnet, y_true_resnet, y_pred_resnet = train_and_evaluate(
    'ResNet50V2', resnet_backbone, resnet_preprocess, ds_train, ds_val,
    fine_tune_layers=50
)

## 8. Compare Models

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'MobileNetV2': metrics_mobile,
    'ResNet50V2':  metrics_resnet,
}).T

print('\n=== Model Comparison (Validation) ===')
print(df.to_string())

best_name  = 'ResNet50V2'  if metrics_resnet['f1'] >= metrics_mobile['f1'] else 'MobileNetV2'
best_model = model_resnet  if best_name == 'ResNet50V2' else model_mobile
best_preprocess = resnet_preprocess if best_name == 'ResNet50V2' else mobilenet_preprocess
print(f'\nBest model: {best_name}')

## 9. Confusion Matrix (Best Model — Val Set)

In [ ]:
y_true = y_true_resnet if best_name == 'ResNet50V2' else y_true_mobile
y_pred = y_pred_resnet if best_name == 'ResNet50V2' else y_pred_mobile

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(np.log1p(cm), cmap='Blues', xticklabels=False, yticklabels=False)
plt.title(f'Confusion Matrix — {best_name} (log scale)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(f'outputs/confusion_matrix_{best_name}.png')
plt.show()

## 10. Final Test Set Evaluation & Predictions

In [ ]:
test_ds = make_pipeline(ds_test, best_preprocess, training=False)

y_true_test, y_pred_test = [], []
for imgs, labels in test_ds:
    preds = best_model.predict(imgs, verbose=0)
    y_pred_test.extend(np.argmax(preds, axis=1))
    y_true_test.extend(labels.numpy())
y_true_test = np.array(y_true_test)
y_pred_test = np.array(y_pred_test)

print(f'=== TEST SET — {best_name} ===')
print(f'Accuracy  : {accuracy_score(y_true_test, y_pred_test):.4f}')
print(f'Precision : {precision_score(y_true_test, y_pred_test, average="macro", zero_division=0):.4f}')
print(f'Recall    : {recall_score(y_true_test, y_pred_test, average="macro", zero_division=0):.4f}')
print(f'Macro F1  : {f1_score(y_true_test, y_pred_test, average="macro", zero_division=0):.4f}')

# Save predictions
pd.DataFrame({'predicted': y_pred_test, 'true': y_true_test}).to_csv(
    'outputs/final_predictions.csv', index=False
)
print('\nSaved: outputs/final_predictions.csv')

## 11. (Optional) Retrain best model on Train + Validation
The official Flowers102 train split has only **10 images/class**. Merging train+val doubles the
data to ~20/class and is allowed (the **test** set is still untouched). This usually gives the
biggest single jump in contest F1. Run this only after you've picked `best_name` above.

In [ ]:
RUN_COMBINED = False  # set True for the final contest model

if RUN_COMBINED:
    combined_raw = ds_train.concatenate(ds_val)
    if best_name == 'ResNet50V2':
        bb = tf.keras.applications.ResNet50V2(include_top=False, weights='imagenet',
                                              input_shape=(IMG_SIZE, IMG_SIZE, 3))
        prep_fn, ft = resnet_preprocess, 50
    else:
        bb = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet',
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3))
        prep_fn, ft = mobilenet_preprocess, 40

    # Use the test set only to report final numbers, never to choose weights.
    final_model, _, _, _ = train_and_evaluate(
        f'{best_name}_combined', bb, prep_fn, combined_raw, ds_test, fine_tune_layers=ft
    )
    print('Combined-data model saved as best_' + best_name + '_combined.keras')